# Chapter 5: Moon Classifier

## Get the data

To get the data we must generate it using `make_moons`

In [35]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

In [36]:
X,y = make_moons(n_samples = 10000,noise = 0.4)

Let's split the data

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,shuffle=True)

In [38]:
print(X_train.shape[0])
print(y_train.shape[0])

8000
8000


## Fine tune model

Let's use the gridsearch

In [39]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

In [40]:
hyperparams = {'max_leaf_nodes': list(range(2, 100)),
    'max_depth': [1, 2, 3, 4, 5, 6],
    'min_samples_split': [2, 3, 4]
    }

grid_cv = GridSearchCV(DecisionTreeClassifier(random_state=42),hyperparams,cv=3)

grid_cv.fit(X_train,y_train)

,estimator,DecisionTreeC...ndom_state=42)
,param_grid,"{'max_depth': [1, 2, ...], 'max_leaf_nodes': [2, 3, ...], 'min_samples_split': [2, 3, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [41]:
best_params = grid_cv.best_params_

dt_clf = DecisionTreeClassifier(**best_params)


In [42]:
from sklearn.metrics import accuracy_score

dt_clf.fit(X_train,y_train)

y_pred =  dt_clf.predict(X_test)
accuracy_score(y_pred=y_pred,y_true=y_test)

0.8495

## Grow a forest

### Split the data in minisets

In [43]:
from sklearn.model_selection import ShuffleSplit

n_trees = 1000
n_instances = 100

mini_sets = []

rs = ShuffleSplit(n_splits=n_trees, test_size= len(X_train) - n_instances,
                  random_state=42)

for mini_train_index, mini_test_index in rs.split(X_train):
    X_mini_train = X_train[mini_train_index]
    y_mini_train = y_train[mini_train_index]
    mini_sets.append((X_mini_train, y_mini_train))

### Train a tree per set

In [44]:
from sklearn.base import clone

# Using the best parameters
forest = [clone(grid_cv.best_estimator_) for _ in range(n_trees)]

scores = []

for tree, (mini_x_train, mini_y_train) in zip(forest,mini_sets):
    tree.fit(mini_x_train,mini_y_train)

    y_pred = tree.predict(X_test)
    scores.append(accuracy_score(y_pred=y_pred,y_true=y_test))
    
np.mean(scores)

np.float64(0.802062)

### Evaluate one instance with all trees

In [45]:
Y_pred = np.empty([n_trees, len(X_test)], dtype=np.uint8)

for tree_index, tree in enumerate(forest):
    Y_pred[tree_index] = tree.predict(X_test)

In [46]:
from scipy.stats import mode

y_pred_majority_votes, n_votes = mode(Y_pred, axis=0)

In [47]:
accuracy_score(y_test, y_pred_majority_votes.reshape([-1]))

0.8595